# 03a — Fixed-log: The Fleet App

The commands from `03_fleet_app_guide.ipynb`, run on the sample data. Uses the complete
model from the `02` examples if you ran one, otherwise the ready-made sample model.

**Before you start:** put your API key in a file named `.env` in this folder, as
`AMYGDA_SDK_API_KEY=your_key_here` (the same file the other examples use).

## Setup

In [ ]:
import glob, os, shutil, sys

# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

# The `amygda-risk` command (run through this kernel's Python if it isn't on PATH)
AMYGDA_RISK = shutil.which("amygda-risk") or f'"{sys.executable}" -m amygda_ops_risk_score.monitor.cli'

# Yours from the 02 examples if you ran one, otherwise the ready-made sample model
MODEL_ZIPS = sorted(glob.glob("outputs/fixed_log/complete_model_*.zip"), key=os.path.getmtime)
MODEL      = MODEL_ZIPS[-1] if MODEL_ZIPS else "sample_data/models/fixed_log_complete_model.zip"
TRAINING   = "sample_data/fixed_log_training.csv"      # earlier logs, used as history
MON_DIR    = "outputs/fixed_log_fleet_app"            # the app's folder
SPEC_FILE  = "outputs/fixed_log_fleet_app.json"       # the layout

os.makedirs("outputs", exist_ok=True)
print("Model:", MODEL)

## 1. See what the app can show

In [ ]:
!{AMYGDA_RISK} panels

## 2. Write the layout

In [ ]:
%%writefile outputs/fixed_log_fleet_app.json
{
  "title": "Baggage stations risk",
  "theme": {"mode": "auto", "accent": "#0b6e4f"},
  "header": {"tiles": ["open_alerts", "assets_high_now", "highest", "last_run"]},
  "tabs": [
    {"title": "Fleet",  "panels": ["since_last_run", "fleet_systems", "ranking"]},
    {"title": "Alerts", "panels": ["alerts"]},
    {"title": "Assets", "panels": ["asset_trend", "asset_systems", "asset_drivers", "asset_logs"]},
    {"title": "Runs",   "panels": ["run_health", "run_table"]}
  ],
  "alerts": {"rules": [
    {"type": "enters_band", "band": "watch"},
    {"type": "rises_by", "points": 8},
    {"type": "run_failed", "runs": 1}
  ]},
  "schedule": {"every": "day", "at": "06:00"}
}

## 3. Preview it

Opens the page in your browser, built from the scores saved in the model zip. Nothing is
deployed.

In [ ]:
!{AMYGDA_RISK} design --spec {SPEC_FILE} --model {MODEL} --open

## 4. Set up the app

Deploys the model, creates the folder and applies the layout. Nothing is scored.

In [ ]:
!{AMYGDA_RISK} init {MON_DIR} --model {MODEL} --name example-baggage-stations \
    --history {TRAINING} --fleet-name "Baggage stations risk" --api-key-env AMYGDA_SDK_API_KEY

In [ ]:
!{AMYGDA_RISK} design {MON_DIR} --spec {SPEC_FILE}

## 5. Open it and add data

The app runs until you stop it, so start it from a terminal in this folder, not from a
notebook cell:

```bash
amygda-risk app outputs/fixed_log_fleet_app
```

It opens at `http://127.0.0.1:8050/`, waiting for data. Press **Add files**, choose
`sample_data/batches/fixed_log/week_01.csv`, and press **Run now**. Add the later weeks one at a
time, or several together, and watch the page change.

## 6. Make it start by itself (optional)

This only shows what would be installed. Add `--install` to do it, and use `--remove` to
take it out again.

In [ ]:
!{AMYGDA_RISK} service {MON_DIR} --name example-baggage-stations

## 7. Where things stand

In [ ]:
!{AMYGDA_RISK} status {MON_DIR}

## Clean up (optional)

Lists what removing the app would do: stop it, delete the deployed model and delete the
folder. Add `--yes` to do it.

In [ ]:
!{AMYGDA_RISK} remove {MON_DIR}